<a href="https://colab.research.google.com/github/DanyRE4020/Concentracion_IA_avanzada/blob/main/Notebook8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Notebook 8 — Covarianza, correlación, eigenvalores, VIF, leverage y Cook


In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
data_2026 = pd.read_csv('/content/contaminantes_2026.csv', sep=',',skiprows=9,encoding="utf-8")
data_2026.columns = ["date", "id_station", "id_parameter", "value", "unit"]
data_2026.head()

,date,id_station,id_parameter,value,unit
0,2026-01-01 00:00:00,ACO,CO,NaN,15.0
1,2026-01-01 00:00:00,ACO,NO,NaN,1.0
2,2026-01-01 00:00:00,ACO,NO2,NaN,1.0
3,2026-01-01 00:00:00,ACO,NOX,NaN,1.0
4,2026-01-01 00:00:00,ACO,O3,NaN,1.0


In [13]:

def _clean_name(x):
    return (str(x).strip().lower()
            .replace('á','a').replace('é','e').replace('í','i')
            .replace('ó','o').replace('ú','u').replace('ñ','n'))

def _find_col(cols, candidates):
    normalized = {_clean_name(c): c for c in cols}
    for cand in candidates:
        if cand in normalized:
            return normalized[cand]
    for c in cols:
        cn = _clean_name(c)
        if any(cand in cn for cand in candidates):
            return c
    return None

def make_wide(df):
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]

    station_col = _find_col(d.columns, ['id_station','station','estacion','cve_est','id_estacion'])
    date_col = _find_col(d.columns, ['date','fecha','datetime','timestamp'])
    hour_col = _find_col(d.columns, ['hour','hora'])
    param_col = _find_col(d.columns, ['id_parameter','parameter','parametro','contaminant','contaminante'])
    value_col = _find_col(d.columns, ['value','valor','concentration','concentracion'])

    if station_col is None or date_col is None:
        raise ValueError(f'No pude detectar las columnas de estación/fecha. Columnas: {list(d.columns)}')

    base_date = pd.to_datetime(d[date_col], errors='coerce')
    if hour_col is not None and _clean_name(hour_col) not in ['datetime','timestamp']:
        h = pd.to_numeric(d[hour_col], errors='coerce')
        extra_day = (h == 24).astype(int)
        h = h.where(h != 24, 0)
        d['datetime'] = base_date.dt.normalize() + pd.to_timedelta(extra_day, unit='D') + pd.to_timedelta(h, unit='h')
    else:
        d['datetime'] = base_date

    d['id_station'] = d[station_col].astype(str).str.strip().str.upper()

    if param_col is not None and value_col is not None:
        d['pollutant'] = d[param_col].astype(str).str.strip().str.upper()
        d['value_num'] = pd.to_numeric(d[value_col], errors='coerce')
        wide = (d.pivot_table(index=['datetime','id_station'], columns='pollutant', values='value_num', aggfunc='mean')
                  .reset_index())
    else:
        wide = d.copy()
        fixed = {}
        for c in wide.columns:
            if c not in ['datetime','id_station']:
                fixed[c] = str(c).strip().upper()
        wide = wide.rename(columns=fixed)

    ren = {}
    for c in wide.columns:
        cu = str(c).upper().replace(' ', '').replace('_','').replace('.','')
        if cu in ['PM25','PM2,5']:
            ren[c] = 'PM2.5'
        elif cu == 'PM10':
            ren[c] = 'PM10'
        elif cu == 'PMCO':
            ren[c] = 'PMCO'
        elif cu in ['NOX','NO2','NO','CO','O3','SO2']:
            ren[c] = cu
    wide = wide.rename(columns=ren)
    return wide.sort_values(['datetime','id_station']).reset_index(drop=True)

wide_raw = make_wide(data_2026)
pollutants = [c for c in ['CO','NO','NO2','NOX','O3','PM10','PM2.5','PMCO','SO2'] if c in wide_raw.columns]
for c in pollutants:
    wide_raw[c] = pd.to_numeric(wide_raw[c], errors='coerce')

print('Contaminantes encontrados:', pollutants)
print('Número de estaciones:', wide_raw['id_station'].nunique())
wide_raw.head()


Contaminantes encontrados: ['CO', 'NO', 'NO2', 'NOX', 'O3', 'PM10', 'PM2.5', 'PMCO', 'SO2']
Número de estaciones: 31


pollutant,datetime,id_station,CO,NO,NO2,NOX,O3,PM10,PM2.5,PMCO,SO2
0,2026-01-01,AJM,0.74,2.0,26.0,28.0,19.0,67.0,53.0,14.0,28.0
1,2026-01-01,ATI,0.70,2.0,21.0,23.0,18.0,90.0,NaN,NaN,22.0
2,2026-01-01,BJU,1.07,NaN,32.0,NaN,17.0,62.0,41.0,21.0,3.0
3,2026-01-01,CAM,NaN,7.0,44.0,52.0,4.0,57.0,49.0,8.0,25.0
4,2026-01-01,CCA,1.12,1.0,32.0,33.0,13.0,NaN,27.0,NaN,3.0


## a) Tabla: un renglón por estación-hora y una columna por contaminante

In [14]:

wide = wide_raw.copy()
cols_show = ['datetime','id_station'] + [c for c in ['CO','NO','NO2','NOX','O3','PM10','PM2.5','PMCO','SO2'] if c in wide.columns]
tabla_estacion_hora = wide[cols_show].copy()
print('Forma:', tabla_estacion_hora.shape)
tabla_estacion_hora.head(10)


Forma: (110510, 11)


pollutant,datetime,id_station,CO,NO,NO2,NOX,O3,PM10,PM2.5,PMCO,SO2
0,2026-01-01,AJM,0.74,2.0,26.0,28.0,19.0,67.0,53.0,14.0,28.0
1,2026-01-01,ATI,0.70,2.0,21.0,23.0,18.0,90.0,NaN,NaN,22.0
2,2026-01-01,BJU,1.07,NaN,32.0,NaN,17.0,62.0,41.0,21.0,3.0
3,2026-01-01,CAM,NaN,7.0,44.0,52.0,4.0,57.0,49.0,8.0,25.0
4,2026-01-01,CCA,1.12,1.0,32.0,33.0,13.0,NaN,27.0,NaN,3.0
5,2026-01-01,CUA,1.05,5.0,28.0,33.0,22.0,55.0,NaN,NaN,28.0
6,2026-01-01,CUT,0.80,2.0,29.0,31.0,9.0,NaN,NaN,NaN,12.0
7,2026-01-01,FAC,0.72,3.0,21.0,24.0,19.0,58.0,NaN,NaN,24.0
8,2026-01-01,GAM,NaN,NaN,53.0,NaN,2.0,98.0,65.0,34.0,NaN
9,2026-01-01,HGM,0.93,2.0,36.0,38.0,12.0,NaN,NaN,NaN,23.0


## b) Covarianza y correlación de CO con NOX. ¿Son distintas?

In [15]:

co_nox = tabla_estacion_hora[['CO','NOX']].copy()
co_nox.loc[co_nox['CO'] < 0, 'CO'] = np.nan
co_nox.loc[co_nox['NOX'] < 0, 'NOX'] = np.nan
co_nox = co_nox.dropna()

cov_co_nox = co_nox['CO'].cov(co_nox['NOX'])
corr_co_nox = co_nox['CO'].corr(co_nox['NOX'])
print('Covarianza CO-NOX =', cov_co_nox)
print('Correlación CO-NOX =', corr_co_nox)
print('Sí son distintas: la covarianza depende de la escala y conserva unidades; la correlación está estandarizada entre -1 y 1.')


Covarianza CO-NOX = 12.8941919470368
Correlación CO-NOX = 0.8695841487862728
Sí son distintas: la covarianza depende de la escala y conserva unidades; la correlación está estandarizada entre -1 y 1.


## c) Eigenvalores de la matriz de correlación, y luego sin faltantes

In [16]:

num_cols = [c for c in ['CO','NO','NO2','NOX','O3','PM10','PM2.5','PMCO','SO2'] if c in tabla_estacion_hora.columns]
M = tabla_estacion_hora[num_cols].copy()
for c in num_cols:
    M.loc[M[c] < 0, c] = np.nan

corr_pairwise = M.corr()
print('Matriz de correlación con pares disponibles:')
display(corr_pairwise)
if corr_pairwise.isna().any().any():
    print('La matriz aún contiene NaN y no se calculan eigenvalores de esta versión.')
else:
    eig_pairwise = np.linalg.eigvalsh(corr_pairwise.values)
    print('Eigenvalores pairwise:', np.round(eig_pairwise, 6))

M_complete = M.dropna()
corr_complete = M_complete.corr()
eig_complete = np.linalg.eigvalsh(corr_complete.values)
print('Casos completos:', len(M_complete))
print('Eigenvalores sin faltantes:', np.round(eig_complete, 6))
print('Eigenvalor mínimo:', eig_complete.min())
print('Suma de eigenvalores:', eig_complete.sum(), '≈', len(num_cols))


Matriz de correlación con pares disponibles:


pollutant,CO,NO,NO2,NOX,O3,PM10,PM2.5,PMCO,SO2
pollutant,,,,,,,,,
CO,1.000000,0.765129,0.764237,0.869584,-0.355109,0.429674,0.404322,0.445880,0.154394
NO,0.765129,1.000000,0.485442,0.942203,-0.395289,0.258270,0.192457,0.291310,0.083600
NO2,0.764237,0.485442,1.000000,0.750043,-0.447556,0.407676,0.339377,0.429936,0.155536
NOX,0.869584,0.942203,0.750043,1.000000,-0.466206,0.352750,0.282163,0.382369,0.120224
O3,-0.355109,-0.395289,-0.447556,-0.466206,1.000000,0.053067,0.160559,0.020505,-0.043795
PM10,0.429674,0.258270,0.407676,0.352750,0.053067,1.000000,0.841947,0.847423,0.217055
PM2.5,0.404322,0.192457,0.339377,0.282163,0.160559,0.841947,1.000000,0.442277,0.229920
PMCO,0.445880,0.291310,0.429936,0.382369,0.020505,0.847423,0.442277,1.000000,0.222941
SO2,0.154394,0.083600,0.155536,0.120224,-0.043795,0.217055,0.229920,0.222941,1.000000


Eigenvalores pairwise: [-5.792000e-03  1.260000e-04  1.426740e-01  4.248110e-01  5.435310e-01
  6.626810e-01  9.270010e-01  1.968940e+00  4.336028e+00]
Casos completos: 14477
Eigenvalores sin faltantes: [1.180000e-04 3.510000e-04 1.225090e-01 3.262130e-01 5.499600e-01
 6.459500e-01 9.221290e-01 1.959048e+00 4.473722e+00]
Eigenvalor mínimo: 0.00011769877284100496
Suma de eigenvalores: 9.0 ≈ 9


## d) PM2.5 contra los demás: R² y VIF; luego sólo con gases

In [17]:

def fit_pm25(predictors, data):
    cols = ['PM2.5'] + predictors
    d = data[cols].copy()
    for c in cols:
        d.loc[d[c] < 0, c] = np.nan
    d = d.dropna()
    X = sm.add_constant(d[predictors])
    y = d['PM2.5']
    m = sm.OLS(y, X).fit()
    vif = pd.DataFrame({
        'variable': X.columns,
        'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })
    return d, m, vif

all_predictors = [c for c in ['CO','NO','NO2','NOX','O3','PM10','PMCO','SO2'] if c in tabla_estacion_hora.columns]
d_all, m_all, vif_all = fit_pm25(all_predictors, tabla_estacion_hora)
print('MODELO 1: PM2.5 ~ todos los demás contaminantes')
print('n =', len(d_all), '| R² =', round(m_all.rsquared,4), '| R² ajustado =', round(m_all.rsquared_adj,4))
display(pd.DataFrame({'coeficiente':m_all.params, 'p_value':m_all.pvalues}))
display(vif_all)

gases = [c for c in ['CO','NO','NO2','NOX','O3','SO2'] if c in tabla_estacion_hora.columns]
d_gas, m_gas, vif_gas = fit_pm25(gases, tabla_estacion_hora)
print('MODELO 2: PM2.5 ~ sólo gases')
print('n =', len(d_gas), '| R² =', round(m_gas.rsquared,4), '| R² ajustado =', round(m_gas.rsquared_adj,4))
display(pd.DataFrame({'coeficiente':m_gas.params, 'p_value':m_gas.pvalues}))
display(vif_gas)


MODELO 1: PM2.5 ~ todos los demás contaminantes
n = 14477 | R² = 0.9984 | R² ajustado = 0.9984


,coeficiente,p_value
const,0.017075,0.187597
CO,0.014916,0.554132
NO,-0.002139,0.793773
NO2,-0.001684,0.837043
NOX,0.001792,0.826633
O3,-0.000007,0.969013
PM10,0.997997,0.000000
PMCO,-0.997368,0.000000
SO2,0.002272,0.001136


,variable,VIF
0,const,1.000000
1,CO,5.736416
2,NO,2651.824792
3,NO2,822.522462
4,NOX,5027.042133
5,O3,1.782303
6,PM10,4.651449
7,PMCO,3.509190
8,SO2,1.103711


MODELO 2: PM2.5 ~ sólo gases
n = 26858 | R² = 0.3103 | R² ajustado = 0.3101


,coeficiente,p_value
const,1.602096,4.566201e-14
CO,12.380579,9.333040e-273
NO,-0.037320,7.789315e-01
NO2,0.237785,7.343444e-02
NOX,0.006488,9.610706e-01
O3,0.161729,0.000000e+00
SO2,0.336962,2.071315e-167


,variable,VIF
0,const,1.000000
1,CO,4.256664
2,NO,2660.420825
3,NO2,717.605492
4,NOX,4696.614850
5,O3,1.362781
6,SO2,1.036953


## e) Matriz H con una estación y una semana

In [18]:

PREFERRED_STATION = 'MER'
base = tabla_estacion_hora[['datetime','id_station','PM2.5'] + gases].copy()
for c in ['PM2.5'] + gases:
    base.loc[base[c] < 0, c] = np.nan

stations = [PREFERRED_STATION] + [s for s in base['id_station'].dropna().unique() if s != PREFERRED_STATION]
selected = None
for st in stations:
    tmp = base[base['id_station'].eq(st)].dropna(subset=['PM2.5'] + gases).sort_values('datetime')
    if len(tmp) < len(gases) + 3:
        continue
    for start in tmp['datetime'].dt.normalize().drop_duplicates():
        end = start + pd.Timedelta(days=7)
        wk = tmp[(tmp['datetime'] >= start) & (tmp['datetime'] < end)].copy()
        if len(wk) >= len(gases) + 3:
            selected = (st, start, end, wk)
            break
    if selected is not None:
        break

if selected is None:
    raise ValueError('No se encontró una estación-semana con suficientes datos completos.')

STATION_H, START, END, week = selected
print('Estación:', STATION_H)
print('Semana:', START.date(), 'a', (END-pd.Timedelta(days=1)).date())
print('n =', len(week))

Xw = sm.add_constant(week[gases]).to_numpy(dtype=float)
yw = week['PM2.5'].to_numpy(dtype=float)
n, p = Xw.shape
XtX_inv = np.linalg.pinv(Xw.T @ Xw)
H = Xw @ XtX_inv @ Xw.T

print('Máximo error |H²-H| =', np.max(np.abs(H @ H - H)))
print('¿H² ≈ H?', np.allclose(H @ H, H, atol=1e-8))
print('tr(H) =', np.trace(H))
print('p =', p, '(incluye intercepto)')
print('¿tr(H) ≈ p?', np.isclose(np.trace(H), p, atol=1e-8))


Estación: MER
Semana: 2026-01-24 a 2026-01-30
n = 161
Máximo error |H²-H| = 4.42638981024146e-13
¿H² ≈ H? True
tr(H) = 6.999999999994984
p = 7 (incluye intercepto)
¿tr(H) ≈ p? True


### Leverage sin construir H

In [19]:

leverage_formula = np.einsum('ij,jk,ik->i', Xw, XtX_inv, Xw)
leverage_H = np.diag(H)
print('Máxima diferencia con diag(H):', np.max(np.abs(leverage_formula - leverage_H)))

lev_table = week[['datetime','id_station']].copy().reset_index(drop=True)
lev_table['leverage'] = leverage_formula
lev_table.sort_values('leverage', ascending=False).head(10)


Máxima diferencia con diag(H): 7.683298441918396e-13


pollutant,datetime,id_station,leverage
144,2026-01-30 07:00:00,MER,0.207556
123,2026-01-29 07:00:00,MER,0.180674
114,2026-01-28 22:00:00,MER,0.151723
139,2026-01-29 23:00:00,MER,0.139704
69,2026-01-27 01:00:00,MER,0.122856
153,2026-01-30 16:00:00,MER,0.119202
113,2026-01-28 21:00:00,MER,0.116663
4,2026-01-24 07:00:00,MER,0.111581
154,2026-01-30 17:00:00,MER,0.096869
138,2026-01-29 22:00:00,MER,0.091666


## f) Cook: ¿los 10 más influyentes son los de más leverage? Reajuste sin ellos

In [20]:

week_model = sm.OLS(yw, Xw).fit()
influence = week_model.get_influence()
cooks_d = influence.cooks_distance[0]

comp = week[['datetime','id_station']].copy().reset_index(drop=True)
comp['leverage'] = leverage_formula
comp['cooks_d'] = cooks_d

idx_cook = set(comp.nlargest(10, 'cooks_d').index)
idx_lev = set(comp.nlargest(10, 'leverage').index)
overlap = idx_cook & idx_lev
print('Coincidencias entre top-10 Cook y top-10 leverage:', len(overlap), 'de 10')
print('Índices coincidentes:', sorted(overlap))
print('Top 10 por Cook:')
display(comp.nlargest(10, 'cooks_d'))
print('Top 10 por leverage:')
display(comp.nlargest(10, 'leverage'))
print('Cook y leverage no miden lo mismo: leverage mide qué tan extremo es X; Cook combina leverage y tamaño del residuo.')


Coincidencias entre top-10 Cook y top-10 leverage: 5 de 10
Índices coincidentes: [113, 123, 138, 139, 144]
Top 10 por Cook:


pollutant,datetime,id_station,leverage,cooks_d
144,2026-01-30 07:00:00,MER,0.207556,0.071511
123,2026-01-29 07:00:00,MER,0.180674,0.063910
76,2026-01-27 08:00:00,MER,0.043808,0.056882
138,2026-01-29 22:00:00,MER,0.091666,0.051760
139,2026-01-29 23:00:00,MER,0.139704,0.047929
113,2026-01-28 21:00:00,MER,0.116663,0.043787
149,2026-01-30 12:00:00,MER,0.047859,0.033187
141,2026-01-30 04:00:00,MER,0.055705,0.032549
6,2026-01-24 09:00:00,MER,0.074607,0.029080
118,2026-01-29 02:00:00,MER,0.072021,0.028144


Top 10 por leverage:


pollutant,datetime,id_station,leverage,cooks_d
144,2026-01-30 07:00:00,MER,0.207556,0.071511
123,2026-01-29 07:00:00,MER,0.180674,0.063910
114,2026-01-28 22:00:00,MER,0.151723,0.014125
139,2026-01-29 23:00:00,MER,0.139704,0.047929
69,2026-01-27 01:00:00,MER,0.122856,0.013002
153,2026-01-30 16:00:00,MER,0.119202,0.008431
113,2026-01-28 21:00:00,MER,0.116663,0.043787
4,2026-01-24 07:00:00,MER,0.111581,0.003823
154,2026-01-30 17:00:00,MER,0.096869,0.014183
138,2026-01-29 22:00:00,MER,0.091666,0.051760


Cook y leverage no miden lo mismo: leverage mide qué tan extremo es X; Cook combina leverage y tamaño del residuo.


In [21]:

keep = ~np.isin(np.arange(len(week)), list(idx_cook))
Xw2 = Xw[keep]
yw2 = yw[keep]
model_reduced = sm.OLS(yw2, Xw2).fit()

coef_names = ['const'] + gases
comparison = pd.DataFrame({
    'coef_original': week_model.params,
    'coef_sin_top10_cook': model_reduced.params,
    'cambio_absoluto': np.abs(model_reduced.params - week_model.params)
}, index=coef_names)

print('R² original:', round(week_model.rsquared, 4))
print('R² sin top 10 Cook:', round(model_reduced.rsquared, 4))
display(comparison)
print('Si los coeficientes o el R² cambian mucho, esos puntos influían de manera importante en el ajuste.')


R² original: 0.6672
R² sin top 10 Cook: 0.7171


,coef_original,coef_sin_top10_cook,cambio_absoluto
const,-3.728063,-6.378238,2.650175
CO,-6.668746,-6.197256,0.471490
NO,1.181796,0.614187,0.567609
NO2,1.744852,1.219429,0.525423
NOX,-1.082869,-0.509947,0.572922
O3,0.264573,0.271632,0.007059
SO2,0.111958,0.140255,0.028296


Si los coeficientes o el R² cambian mucho, esos puntos influían de manera importante en el ajuste.
